In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
from model.tqs import SinusoidalPositionalEncoding

## Sinusoidal Position Encoding


In [ ]:
pos_enc = SinusoidalPositionalEncoding(d_model=20, max_len=20, device=torch.device("cpu"))

test_buf = torch.zeros(10, 1, 20, device=torch.device("cpu"))
test_buf = pos_enc(test_buf).squeeze(1)

# test_buf: (seq_len, 1, d_model), squeeze to (seq_len, d_model)
heatmap = test_buf.detach().cpu().numpy()

plt.figure(figsize=(8, 4))
plt.imshow(heatmap, aspect="auto", cmap="RdBu", interpolation="nearest")
plt.colorbar(label="Encoding Value")
plt.xlabel("Embedding dimension")
plt.ylabel("Sequence position")
plt.title("Sinusoidal Positional Encoding Heatmap")
plt.show()


## Map Into Embedding Space


In [ ]:
from hamiltonian.hamiltonian import Hamiltonian
from model.tqs import TransformerQuantumState

device = torch.device("cpu")

# 1D transverse-field Ising: 1 system dimension, 2 physical params (J coupling, h field)
system_dim = torch.tensor([2.0])
phys_params = torch.tensor([1.0, 0.5])  # J=1, h=0.5

ham = Hamiltonian(n_params=2, system_dim=system_dim, phys_params=phys_params)

d_model = 20
max_len = 20
batch_size = 4

model = TransformerQuantumState(d_model, max_len, batch_size, ham, device)
model.eval()

with torch.no_grad():
    buf = model.init_spin_buffer()
    out = model(buf)

print(f"input  shape: {buf.shape}  (seq, batch, token_width)")
print(f"output shape: {out.shape}  (seq, batch, d_model)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].imshow(buf[:, 0, :].numpy(), aspect="auto", cmap="RdBu", interpolation="nearest")
axes[0].set_xlabel("Token dimension")
axes[0].set_ylabel("Sequence position")
axes[0].set_title("Raw spin buffer (batch element 0)")

im = axes[1].imshow(out[:, 0, :].numpy(), aspect="auto", cmap="RdBu", interpolation="nearest")
axes[1].set_xlabel("Embedding dimension")
axes[1].set_ylabel("Sequence position")
axes[1].set_title("After embedding + positional encoding")

fig.colorbar(im, ax=axes, label="Activation")
plt.show()